In [1]:
import tensorflow as tf
import numpy as np
from PIL import Image, ImageTk
import cv2
import screen_brightness_control as sbc
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
import ctypes
import tkinter as tk
from tkinter import ttk
import os

# Load the models
# Replace with your actual model paths
h1 = tf.keras.models.load_model('D:/pro/New folder2/model.keras')
d = tf.keras.models.load_model('D:/pro/New folder/modelF.keras')

def set_screen_brightness(level):
    """
    تغيير درجة سطوع الشاشة إلى مستوى محدد (0-100).
    يتطلب صلاحيات المسؤول في بعض الأحيان.
    """
    try:
        sbc.set_brightness(level)
        print(f"تم زياده سطوع الشاشة إلى: {level}%")
    except Exception as e:
        print(f"خطأ في تغيير سطوع الشاشة: {e}")
        print("قد تحتاج إلى تشغيل البرنامج كمسؤول (Run as administrator).")

def get_current_brightness():
    """
    الحصول على درجة السطوع الحالية للشاشة.
    """
    try:
        current_brightness = sbc.get_brightness()
        if current_brightness:
            # get_brightness can return a list of brightness values for multiple monitors
            # We take the first one
            print(f"درجة السطوع الحالية: {current_brightness[0]}%")
            return current_brightness[0]
        else:
            print("لم يتمكن من الحصول على درجة السطوع الحالية.")
            return None
    except Exception as e:
        print(f"خطأ في الحصول على درجة السطوع: {e}")
        return None

def set_system_volume(level):
    """
    تغيير درجة صوت النظام الرئيسية (0-100).
    """
    try:
        devices = AudioUtilities.GetSpeakers()
        interface = devices.Activate(
            IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
        volume = ctypes.cast(interface, ctypes.POINTER(IAudioEndpointVolume))
        
        # level is 0-100, SetMasterVolumeLevelScalar expects 0.0-1.0
        volume.SetMasterVolumeLevelScalar(level / 100.0, None)
        print(f"تم ضبط مستوى الصوت إلى: {level}%")

    except Exception as e:
        print(f"خطأ في تغيير مستوى الصوت: {e}")
        print("تأكد من تثبيت 'pycaw' و 'comtypes' بشكل صحيح.")
        print("تأكد أيضاً من أن جهاز الصوت الافتراضي يعمل.")

def get_current_volume():
    """
    الحصول على درجة صوت النظام الحالية (0-100).
    """
    try:
        devices = AudioUtilities.GetSpeakers()
        interface = devices.Activate(
            IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
        volume = ctypes.cast(interface, ctypes.POINTER(IAudioEndpointVolume))
        
        # Get volume scalar (0.0-1.0) and convert to 0-100
        current_volume = int(volume.GetMasterVolumeLevelScalar() * 100)
        print(f"مستوى الصوت الحالي: {current_volume}%")
        return current_volume

    except Exception as e:
        print(f"خطأ في الحصول على مستوى الصوت: {e}")
        print("تأكد من تثبيت 'pycaw' و 'comtypes' بشكل صحيح.")
        print("تأكد أيضاً من أن جهاز الصوت الافتراضي يعمل.")
        return None

def preprocess_image_for_model_h1(image_array):
    """
    معالجة الصورة لنموذج h1 (توقع الأرقام 0-5).
    يتوقع النموذج صورة رمادية 64x64.
    """
    img_pil = Image.fromarray(image_array).resize((64, 64))
    img_gray = img_pil.convert('L')
    img_array = np.array(img_gray)
    img_array = 255 - img_array  # Invert colors if necessary for your model
    img_array = np.expand_dims(img_array, axis=0)
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = img_array / 255.0
    return img_array

def preprocess_image_for_model_d(image_array):
    """
    معالجة الصورة لنموذج d (توقع التصنيف 0-2).
    يتوقع النموذج صورة رمادية 64x64.
    """
    img_pil = Image.fromarray(image_array).resize((64, 64))
    img_gray = img_pil.convert('L')
    img_array = np.array(img_gray)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = img_array / 255.0
    return img_array

In [2]:
import tkinter as tk
from tkinter import filedialog, Toplevel, Label
import numpy as np
from PIL import Image, ImageTk
import screen_brightness_control as sbc
import tensorflow as tf
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
import ctypes
import os

class SystemController:
    def __init__(self):
        self.volume_interface = self._get_volume_interface()

    def _get_volume_interface(self):
        try:
            devices = AudioUtilities.GetSpeakers()
            interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
            return ctypes.cast(interface, ctypes.POINTER(IAudioEndpointVolume))
        except Exception as e:
            print(f"خطأ في تهيئة واجهة الصوت (pycaw): {e}")
            return None

    def set_brightness(self, level):
        try:
            sbc.set_brightness(level)
            print(f"تم تغيير سطوع الشاشة إلى: {level}%")
        except Exception as e:
            print(f"خطأ في تغيير سطوع الشاشة: {e}")

    def get_brightness(self):
        try:
            return sbc.get_brightness()[0]
        except Exception as e:
            print(f"خطأ في الحصول على درجة السطوع: {e}")
            return 50

    def set_volume(self, level):
        if self.volume_interface:
            try:
                self.volume_interface.SetMasterVolumeLevelScalar(level / 100.0, None)
                print(f"تم تغيير مستوى الصوت إلى: {level}%")
            except Exception as e:
                print(f"خطأ في تغيير مستوى الصوت: {e}")

    def get_volume(self):
        if self.volume_interface:
            try:
                return int(self.volume_interface.GetMasterVolumeLevelScalar() * 100)
            except Exception as e:
                print(f"خطأ في الحصول على مستوى الصوت: {e}")
        return 50

class MainApp:
    def __init__(self, master):
        self.master = master
        self.system_controller = SystemController()
        
        self.background_image_path = "C:/Users/User/Downloads/pexels-kindelmedia-8566526.jpg"

        self._load_models()
        self._setup_ui()
        self.master.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # h1 = tf.keras.models.load_model('D:/pro/New folder2/model.keras')
        # d = tf.keras.models.load_model('D:/pro/New folder/modelF.keras')


    def _load_models(self):
        try:
            # !!! الرجاء التأكد من صحة هذا المسار
            self.model_d = tf.keras.models.load_model('D:/pro/New folder/modelF.keras')
            print("نموذج MainApp تم تحميله بنجاح.")
        except Exception as e:
            print(f"خطأ في تحميل نموذج MainApp: {e}")
            self.model_d = None

    def _setup_ui(self):
        self.master.title("welcom to user 1")
        self.master.geometry("1000x700")
        try:
            original_image = Image.open(self.background_image_path)
            resized_image = original_image.resize((1000, 700), Image.Resampling.LANCZOS)
            self.background_photo = ImageTk.PhotoImage(resized_image)
            self.background_label = tk.Label(self.master, image=self.background_photo)
            self.background_label.place(x=0, y=0, relwidth=1, relheight=1)
        except Exception as e:
            self.master.configure(bg="#8CC3D2")
            error_msg = f"لم يتم العثور على صورة الخلفية!\nتحقق من المسار: {self.background_image_path}"
            #  (FIX) Use .place() to avoid geometry manager conflict
            error_label = Label(self.master, text=error_msg, font=("Arial", 14), fg="red", bg="white")
            error_label.place(relx=0.5, rely=0.1, anchor="center")
            print(f"خطأ تحميل خلفية MainApp: {e}")

        # This part uses .grid(), so the error label MUST NOT use .pack()
        self.master.grid_rowconfigure(0, weight=1)
        self.master.grid_columnconfigure(0, weight=1)
        
        image_widgets = self._create_content_panel(
            self.master, "prediction from image", "choose image for prediction", self.choose_image_for_prediction
        )
        self.input_image_label = image_widgets['display_area']
        self.prediction_image_text_label = image_widgets['result_label']
        image_widgets['frame'].grid(row=0, column=0, sticky="nsew", padx=20, pady=20)


    def _create_content_panel(self, parent, title_text, button_text, button_command):
        frame = tk.Frame(parent, bg="#5199DC", padx=20, pady=20, bd=5, relief="groove")
        frame.grid_rowconfigure(2, weight=1)
        frame.grid_columnconfigure(0, weight=1)
        tk.Label(frame, text=title_text, font=("Arial", 20, "bold"), bg="#A6D5E0", fg="#1F4597").grid(row=0, column=0, pady=10)
        result_label = tk.Label(frame, text="النتيجة: -", font=("Arial", 16), bg="#A6D5E0", fg="#1F4597")
        result_label.grid(row=1, column=0, pady=10)
        display_area = tk.Label(frame, bg="#A6D5E0", borderwidth=2, relief="groove")
        display_area.grid(row=2, column=0, pady=10, sticky="nsew", padx=20)
        tk.Button(frame, text=button_text, command=button_command, bg="#3F84CD", fg="white", font=("Arial", 14, "bold")).grid(row=3, column=0, pady=20)
        return {"frame": frame, "display_area": display_area, "result_label": result_label}



    def choose_image_for_prediction(self):
        image_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
        if image_path:
            self.process_selected_image(image_path)
            
    def process_selected_image(self, img_path):
        if not self.model_d:
            self.prediction_image_text_label.config(text="خطأ: نموذج التوقع D غير محمل.", fg="red")
            return
        try:
            img_pil = Image.open(img_path)
            processed_img = self._preprocess_image(np.array(img_pil))
            prediction = np.argmax(self.model_d.predict(processed_img, verbose=0))
            self.prediction_image_text_label.config(text=f"نتيجة توقع الصورة: {prediction}")
            self._display_image(img_pil, self.input_image_label, (400, 400))
            self._execute_prediction_action(prediction)
        except Exception as e:
            self.prediction_image_text_label.config(text=f"خطأ في معالجة الصورة: {e}", fg="red")

    def _preprocess_image(self, image_array):
        img_pil = Image.fromarray(image_array).resize((64, 64)).convert('L')
        img_array = np.array(img_pil) / 255.0
        return np.expand_dims(img_array, axis=(0, -1))

    def _execute_prediction_action(self, prediction_class):
        actions = {0: lambda: self.system_controller.set_brightness(70),
                   1: lambda: self.system_controller.set_brightness(30),
                   2: lambda: self.system_controller.set_volume(70)}
        action_to_run = actions.get(prediction_class)
        if action_to_run:
            action_to_run()

    def _display_image(self, pil_image, target_label, size):
        if size[0] < 1 or size[1] < 1: size = (100, 100)
        resized_image = pil_image.resize(size, Image.Resampling.LANCZOS)
        imgtk = ImageTk.PhotoImage(image=resized_image)
        target_label.imgtk = imgtk
        target_label.config(image=imgtk)

    def on_closing(self):
        self.master.destroy()

class ThirdApp:
    def __init__(self, master):
        self.master = master
        master.title("welcom to user 2")
        master.geometry("800x600")
        tk.Label(master, text="أهلاً بك أيها المستخدم الثاني!", font=("Arial", 24)).pack(pady=50)

    def on_closing(self):
        self.master.destroy()

class InitialApp:
    def __init__(self, master, main_app_class, third_app_class):
        self.master = master
        self.main_app_class = main_app_class
        self.third_app_class = third_app_class
        self.password_one_sequence = [2, 4, 3, 2]
        self.password_two_sequence = [1, 5, 0, 1]
        self.reset_state()
        
        self._load_initial_models()
        self._setup_ui()
        self.master.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _load_initial_models(self):
        try:
            # !!! الرجاء التأكد من صحة هذه المسارات
            # self.model_h1 = tf.keras.models.load_model('D:/pro/New folder2/model.keras')
            self.model_h1 = tf.keras.models.load_model('D:/pro/New folder2/model.keras')
            self.model_d = tf.keras.models.load_model('D:/pro/New folder/modelF.keras')
            # self.model_d = tf.keras.models.load_model('D:/pro/New folder/modelF.keras')
            
            print("نماذج InitialApp تم تحميلها بنجاح.")
        except Exception as e:
            print(f"خطأ في تحميل نماذج InitialApp: {e}")
            self.model_h1 = None
            self.model_d = None

    def _setup_ui(self):
        self.master.title("الواجهة الأولية - التحقق من الدخول")
        self.master.geometry("1000x700")
        
        # !!! الرجاء التأكد من صحة هذا المسار
        background_image_path = "C:/Users/User/Downloads/pexels-kindelmedia-8566526.jpg"
        try:
            original_image = Image.open(background_image_path)
            resized_image = original_image.resize((1000, 700), Image.Resampling.LANCZOS)
            self.background_photo = ImageTk.PhotoImage(resized_image)
            self.background_label = tk.Label(self.master, image=self.background_photo)
            self.background_label.place(x=0, y=0, relwidth=1, relheight=1)
        except Exception as e:
            self.master.configure(bg="#79AAEA")
            error_msg = f"لم يتم العثور على صورة الخلفية!\nتحقق من المسار: {background_image_path}"
            # (FIX) Use .place() to avoid geometry manager conflict
            error_label = Label(self.master, text=error_msg, font=("Arial", 14), fg="red", bg="white")
            error_label.place(relx=0.5, rely=0.1, anchor="center")
            print(f"خطأ تحميل خلفية InitialApp: {e}")

        # This part uses .place(), so the error label can also use .place()
        center_frame = tk.Frame(self.master, bg="#5199DC", padx=30, pady=30, bd=5, relief="groove")
        center_frame.place(relx=0.5, rely=0.5, anchor="center")

        self.instruction_label = tk.Label(center_frame, text="أدخل 4 صور لتكوين كلمة السر", font=("Arial", 24, "bold"), bg="#A6D5E0", fg="#1F4597")
        self.instruction_label.pack(pady=(0, 20))

        self.upload_button = tk.Button(center_frame, text="اختر الصورة", command=self.upload_image, bg="#3F84CD", fg="white", font=("Arial", 18, "bold"), relief="raised", bd=5)
        self.upload_button.pack(pady=10, ipady=15, ipadx=50)

        self.all_predictions_label = tk.Label(center_frame, text="التوقعات المتتالية: []", font=("Arial", 16, "bold"), bg="#A6D5E0", fg="#1F4597")
        self.all_predictions_label.pack(pady=20)
        
        self.image_display_label = tk.Label(center_frame, bg="#A6D5E0", borderwidth=3, relief="groove", width=100, height=100)
        self.image_display_label.pack(pady=10)
        self.image_display_label.pack_propagate(False)

    def _display_image(self, pil_image, target_label, size):
        if size[0] < 1 or size[1] < 1: size = (100, 100)
        display_size = (target_label.winfo_width(), target_label.winfo_height())
        if display_size[0] < 2 or display_size[1] < 2: display_size = size
        
        resized_image = pil_image.resize(display_size, Image.Resampling.LANCZOS)
        imgtk = ImageTk.PhotoImage(image=resized_image)
        target_label.imgtk = imgtk
        target_label.config(image=imgtk)

    def upload_image(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png *.gif")])
        if file_path:
            self.process_image(file_path)

    def process_image(self, img_path):
        if not self.model_h1 or not self.model_d:
            self.instruction_label.config(text="خطأ: النماذج غير محملة.", fg="red")
            return
        
        try:
            pil_image = Image.open(img_path)
            self._display_image(pil_image, self.image_display_label, (250, 250))
            
            img = tf.keras.preprocessing.image.load_img(img_path, target_size=(64, 64), color_mode='grayscale')
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0
            
            prediction = None
            if self.stage == "entering_sequence":
                prediction = np.argmax(self.model_h1.predict(img_array, verbose=0))
                self.handle_sequence_stage(prediction)
            elif self.stage == "awaiting_final_image":
                prediction = np.argmax(self.model_d.predict(img_array, verbose=0))
                self.handle_final_image_stage(prediction)
        except Exception as e:
            self.instruction_label.config(text=f"حدث خطأ: {e}", fg="red")

    def handle_sequence_stage(self, prediction):
        self.predictions.append(prediction)
        self.all_predictions_label.config(text=f"التوقعات المتتالية: {self.predictions}")
        self.current_image_index += 1
        if self.current_image_index >= 4:
            self.check_sequences()

    def handle_final_image_stage(self, prediction):
        if (self.validated_password == 1 and prediction == 3) or \
           (self.validated_password == 2 and prediction == 4):
            success_message = "أهلاً بك في المستخدم الاول" if self.validated_password == 1 else "أهلاً بك في المستخدم الثاني"
            self.instruction_label.config(text=f"التحقق صحيح! {success_message}...", fg="green")
            self.master.after(2000, self.launch_app)
        else:
            self.instruction_label.config(text=f"فشل التحقق (التوقع كان {prediction}). حاول مجدداً.", fg="red")
            self.master.after(3000, self.reset_state_and_ui)

    def check_sequences(self):
        self.upload_button.config(state=tk.DISABLED)
        if self.predictions == self.password_one_sequence:
            self.validated_password = 1
            self.stage = "awaiting_final_image"
            self.instruction_label.config(text="كلمة السر 1 صحيحة! أدخل صورة التحقق النهائية.", fg="blue")
            self.upload_button.config(text="اختر صورة التحقق", state=tk.NORMAL)
        elif self.predictions == self.password_two_sequence:
            self.validated_password = 2
            self.stage = "awaiting_final_image"
            self.instruction_label.config(text="كلمة السر 2 صحيحة! أدخل صورة التحقق النهائية.", fg="blue")
            self.upload_button.config(text="اختر صورة التحقق", state=tk.NORMAL)
        else:
            self.instruction_label.config(text="تسلسل خاطئ. سيتم إعادة الضبط.", fg="red")
            self.master.after(3000, self.reset_state_and_ui)

    def reset_state_and_ui(self):
        self.reset_state()
        self.instruction_label.config(text="أدخل 4 صور لتكوين كلمة السر.", fg="#1F4597")
        self.all_predictions_label.config(text="التوقعات المتتالية: []")
        self.upload_button.config(text="اختر الصورة", state=tk.NORMAL)
        self.image_display_label.config(image='')

    def reset_state(self):
        self.predictions = []
        self.current_image_index = 0
        self.stage = "entering_sequence"
        self.validated_password = 0

    def launch_app(self):
        self.master.destroy()
        root = tk.Tk()
        app_class = self.main_app_class if self.validated_password == 1 else self.third_app_class
        app = app_class(root)
        root.mainloop()

    def on_closing(self):
        self.master.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    app = InitialApp(root, MainApp, ThirdApp)
    root.mainloop()





نماذج InitialApp تم تحميلها بنجاح.


In [3]:
import tkinter as tk
from tkinter import filedialog, Toplevel, Label
import numpy as np
from PIL import Image, ImageTk
import cv2
import screen_brightness_control as sbc
import tensorflow as tf
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
import ctypes
import os

# =================================================================================
# الفئة 1: التحكم في وظائف النظام (الصوت والسطوع)
# =================================================================================
class SystemController:
    def __init__(self):
        try:
            devices = AudioUtilities.GetSpeakers()
            interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
            self.volume_interface = ctypes.cast(interface, ctypes.POINTER(IAudioEndpointVolume))
        except Exception as e:
            self.volume_interface = None
            print(f"خطأ في تهيئة واجهة الصوت: {e}")

    def set_brightness(self, level):
        try:
            sbc.set_brightness(max(0, min(100, level)))
            print(f"تم تغيير سطوع الشاشة إلى: {level}%")
        except Exception as e:
            print(f"خطأ في تغيير سطوع الشاشة: {e}")

    def get_brightness(self):
        try:
            return sbc.get_brightness()[0]
        except Exception as e:
            print(f"خطأ في الحصول على درجة السطوع: {e}")
            return 50

    def set_volume(self, level):
        if self.volume_interface:
            try:
                scalar_volume = max(0.0, min(1.0, level / 100.0))
                self.volume_interface.SetMasterVolumeLevelScalar(scalar_volume, None)
                print(f"تم تغيير مستوى الصوت إلى: {int(scalar_volume * 100)}%")
            except Exception as e:
                print(f"خطأ في تغيير مستوى الصوت: {e}")

    def get_volume(self):
        if self.volume_interface:
            try:
                return int(self.volume_interface.GetMasterVolumeLevelScalar() * 100)
            except Exception as e:
                print(f"خطأ في الحصول على مستوى الصوت: {e}")
        return 50

# =================================================================================
# الفئة 2: التطبيق الرئيسي (تم تصغير الواجهة بنسبة 45%)
# =================================================================================
class MainApp2:
    def __init__(self, master):
        self.master = master
        self.system_controller = SystemController()
        self.user_predictions_history = []
        self.info_window = None
        self.background_photo = None

        self._load_models()
        self._setup_ui()

        self.master.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _load_models(self):
        """
        تحميل نموذج الذكاء الاصطناعي.
        """
        model_path = 'D:/pro/New folder/modelF.keras'
        try:
            if not os.path.exists(model_path):
                print(f"خطأ: ملف النموذج غير موجود في المسار: {model_path}")
                self.model_d = None
                return
            self.model_d = tf.keras.models.load_model(model_path)
            print("تم تحميل النموذج بنجاح.")
        except Exception as e:
            print(f"خطأ في تحميل نموذج TensorFlow: {e}")
            self.model_d = None

    def _setup_ui(self):
        """
        إعداد وتنسيق واجهة المستخدم بعد تصغيرها بنسبة 45%.
        """
        self.master.title("التحكم بالسطوع والصوت عبر التنبؤ")
        
        # تم التعديل: تحديد حجم النافذة ليكون 55% من الحجم الأصلي
        window_width = 1056
        window_height = 594
        self.master.geometry(f"{window_width}x{window_height}")
        self.master.resizable(False, False)

        # 1. إعداد الخلفية
        background_image_path = 'C:/Users/User/Downloads/pexels-kindelmedia-8566526.jpg'
        try:
            original_image = Image.open(background_image_path)
            resized_image = original_image.resize((window_width, window_height), Image.Resampling.LANCZOS)
            self.background_photo = ImageTk.PhotoImage(resized_image)

            background_label = tk.Label(self.master, image=self.background_photo)
            background_label.place(x=0, y=0, relwidth=1, relheight=1)
        except (FileNotFoundError, Exception) as e:
            print(f"خطأ في تحميل صورة الخلفية: {e}. سيتم استخدام لون ثابت.")
            self.master.configure(bg="#79AAEA")

        # 2. إضافة العناصر بأحجام مصغرة (55% من الحجم الأصلي)
        # العنوان الرئيسي
        title_label = tk.Label(self.master, text="التنبؤ من الصور", font=("Arial", 15, "bold"), bg="#A6D5E0", fg="#1F4597")
        title_label.place(relx=0.5, rely=0.15, anchor="center")

        # زر التنبؤ
        predict_button = tk.Button(
            self.master, 
            text="اختر صورة وتنبأ", 
            command=self.choose_image_for_prediction,
            bg="#3F84CD", 
            fg="white", 
            font=("Arial", 11, "bold"), 
            relief="raised", 
            bd=4
        )
        predict_button.place(relx=0.5, rely=0.35, anchor="center", width=165, height=44)

        # مكان عرض نص النتيجة
        self.result_text_label = tk.Label(self.master, text="النتيجة: -", font=("Arial", 10, "bold"), bg="#A6D5E0", fg="#1F4597")
        self.result_text_label.place(relx=0.5, rely=0.55, anchor="center")

        # مكان عرض الصورة
        self.image_display_label = tk.Label(self.master, bg="#A6D5E0", borderwidth=2, relief="groove")
        self.image_display_label.place(relx=0.5, rely=0.8, anchor="center", width=138, height=138)

    def choose_image_for_prediction(self):
        image_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
        if image_path:
            self.process_selected_image(image_path)

    def process_selected_image(self, img_path):
        if not self.model_d:
            self.result_text_label.config(text="خطأ: نموذج التوقع غير محمل.", fg="red")
            return

        try:
            img_pil = Image.open(img_path)
            processed_img = self._preprocess_image(np.array(img_pil))
            prediction = np.argmax(self.model_d.predict(processed_img, verbose=0))
            
            self.user_predictions_history.append(prediction)
            self.result_text_label.config(text=f"نتيجة توقع الصورة: {prediction}", fg="#1F4597")
            
            # عرض الصورة المصغرة
            self._display_image(img_pil, self.image_display_label, (132, 132))
            
            self._execute_prediction_action(prediction)
        except Exception as e:
            self.result_text_label.config(text=f"خطأ في معالجة الصورة: {e}", fg="red")

    def _preprocess_image(self, image_array):
        img_pil = Image.fromarray(image_array).resize((64, 64)).convert('L')
        img_array = np.array(img_pil) / 255.0
        return np.expand_dims(img_array, axis=(0, -1))

    def _execute_prediction_action(self, prediction_class):
        action_map = {
            0: lambda: self.system_controller.set_brightness(self.system_controller.get_brightness() - 20),
            2: lambda: self.system_controller.set_volume(self.system_controller.get_volume() + 20),
            5: self.close_info_window,
            6: self.open_info_window
        }
        
        action_message = {
            0: "تم تنفيذ: خفض السطوع",
            2: "تم تنفيذ: زيادة الصوت",
            5: "تم تنفيذ: إغلاق نافذة المعلومات",
            6: "تم تنفيذ: فتح نافذة المعلومات"
        }

        if prediction_class in action_map:
            action_map[prediction_class]()
            print(action_message[prediction_class])
        else:
            print(f"لا يوجد إجراء معرف للتنبؤ رقم {prediction_class}.")

        if self.info_window:
            self.update_info_window()

    def open_info_window(self):
        if self.info_window and self.info_window.winfo_exists():
            self.info_window.lift()
            return

        self.info_window = Toplevel(self.master)
        self.info_window.title("المعلومات")
        # تعديل حجم نافذة المعلومات
        self.info_window.geometry("220x192")
        self.info_window.configure(bg="#93DBEE")
        self.info_window.protocol("WM_DELETE_WINDOW", self.close_info_window)
        self.update_info_window()

    def update_info_window(self):
        if not (self.info_window and self.info_window.winfo_exists()):
            return

        for widget in self.info_window.winfo_children():
            widget.destroy()

        current_brightness = self.system_controller.get_brightness()
        current_volume = self.system_controller.get_volume()

        # تعديل أحجام الخطوط في نافذة المعلومات
        Label(self.info_window, text="الحالة الحالية", font=("Arial",14, "bold"), bg="#E0F2F7", fg="#1F4597").pack(pady=4)
        Label(self.info_window, text=f"السطوع: {current_brightness}%", font=("Arial", 14), bg="#E0F2F7", fg="#333333").pack(pady=1)
        Label(self.info_window, text=f"الصوت: {current_volume}%", font=("Arial", 14), bg="#E0F2F7", fg="#333333").pack(pady=1)

        Label(self.info_window, text="سجل التنبؤات", font=("Arial", 14, "bold"), bg="#E0F2F7", fg="#1F4597").pack(pady=(10, 6))
        
        history_text = ", ".join(map(str, self.user_predictions_history)) if self.user_predictions_history else "لا توجد"
        Label(self.info_window, text=f"[{history_text}]", font=("Arial", 14), bg="#E0F2F7", fg="#333333", wraplength=192).pack()

    def close_info_window(self):
        if self.info_window:
            self.info_window.destroy()
            self.info_window = None

    def _display_image(self, pil_image, target_label, size):
        try:
            resized_image = pil_image.resize(size, Image.Resampling.LANCZOS)
            imgtk = ImageTk.PhotoImage(image=resized_image)
            target_label.imgtk = imgtk
            target_label.config(image=imgtk)
        except Exception as e:
            print(f"خطأ في عرض الصورة: {e}")

    def on_closing(self):
        self.master.destroy()

# =================================================================================
# نقطة انطلاق البرنامج الرئيسية
# =================================================================================
# if __name__ == "__main__":
#     root = tk.Tk()
#     app = MainApp2(root)
#     root.mainloop()

In [4]:
# from tkinter import filedialog
# import tkinter as tk
# from PIL import Image, ImageTk
# import numpy as np
# import tensorflow as tf

# # --- ملاحظة: تحميل النماذج ---
# # تأكد من تحميل النموذجين 'h1' و 'd' هنا قبل تشغيل الواجهة الرسومية
# # على سبيل المثال:
# # h1 = tf.keras.models.load_model('path/to/your/h1_model.h5')
# # d = tf.keras.models.load_model('path/to/your/d_model.h5')


# class InitialApp:
#     def __init__(self, master, main_app_class, third_app_class):
#         self.master = master
#         self.main_app_class = main_app_class
#         self.third_app_class = third_app_class 

#         # --- تعريف كلمات السر ---
#         self.password_one_sequence = [2, 4, 3, 2]
#         self.password_two_sequence = [1, 5, 0, 1] # كلمة سر جديدة ومختلفة

#         master.title("الواجهة الأولية - التحقق من الدخول")
#         master.geometry("800x600")
#         master.configure(bg="#79AAEA")
           
#         self.predictions = []
#         self.current_image_index = 0
#         self.image_paths = [None] * 4 # To store paths for 4 images
#         self.image_display_labels = [] # To store labels for displaying images

#         # Configure grid layout for the master window
#         master.grid_rowconfigure(0, weight=1)
#         master.grid_rowconfigure(1, weight=0) # For instruction_label
#         master.grid_rowconfigure(2, weight=0) # For prediction_result_label
#         master.grid_rowconfigure(3, weight=0) # For all_predictions_label
#         master.grid_rowconfigure(4, weight=0) # For upload_button
#         master.grid_rowconfigure(5, weight=2) # For image_sequence_frame
#         master.grid_columnconfigure(0, weight=1)


#         # --- إدارة الحالة ---
#         self.reset_state()

#         # --- إعداد الواجهة ---
#         self._setup_ui()
#         self.master.protocol("WM_DELETE_WINDOW", self.on_closing)

#     def _setup_ui(self):
#         background_image_path = 'C:/Users/User/Downloads/pexels-kindelmedia-8566526.jpg' # مثال: ضع الصورة في نفس مجلد الكود
        
#         try:
#             # تحميل الصورة وتغيير حجمها لتناسب النافذة
#             original_image = Image.open(background_image_path)
#             resized_image = original_image.resize((800, 600), Image.LANCZOS)
#             self.background_photo = ImageTk.PhotoImage(resized_image)

#             # إنشاء Label لوضع الصورة كخلفية
#             self.background_label = tk.Label(self.master, image=self.background_photo)
#             self.background_label.place(x=0, y=0, relwidth=1, relheight=1) # لملء النافذة بالكامل

#             # لجعل العناصر الأخرى تظهر فوق الخلفية، نحتاج إلى وضعها في Frame أو التأكد من إعداد `master.grid` بشكل صحيح
#             # وإعدادات `bg` للعناصر الفرعية لتكون شفافة أو مطابقة للخلفية
#             # يمكننا أيضاً استخدام طريقة `place` بدلاً من `grid` للعناصر إذا أردنا تحكماً دقيقاً
#             # في هذا التعديل، سنبقي على `grid` ونغير لون الخلفية للعناصر لتكون شبه شفافة أو ننسقها مع الخلفية.

#         except FileNotFoundError:
#             print(f"خطأ: لم يتم العثور على صورة الخلفية في المسار: {background_image_path}")
#             # في حالة عدم وجود الصورة، يمكن تعيين لون خلفية بديل
        
#             self.master.configure(bg="#79AAEA") 
#             self.background_label = None # لضمان عدم وجود كائن خلفية خاطئ
#         except Exception as e:
#             print(f"حدث خطأ أثناء تحميل صورة الخلفية: {e}")
#             self.master.configure(bg="#79AAEA")
#             self.background_label = None
#         # # إعداد الشبكة وتوزيع العناصر في الواجهة
#         # self.instruction_label = tk.Label(self.master, text="أدخل 4 صور لتكوين كلمة السر.", font=("Arial", 18, "bold"), bg="#79AAEA", fg="#225E9E", pady=10)
#         # self.instruction_label.grid(row=1, column=0, pady=(30,10), sticky="ew")

#         # self.all_predictions_label = tk.Label(self.master, text="التوقعات المتتالية: []", font=("Arial", 14), bg="#79AAEA")
#         # self.all_predictions_label.grid(row=3, column=0, pady=5, sticky="ew")

#         # self.upload_button = tk.Button(self.master, text="اختر الصورة", command=self.upload_image, font=("Arial", 14, "bold"))
#         # self.upload_button.grid(row=4, column=0, pady=(20, 30))
        
        

#         self.instruction_label = tk.Label(self.master, text="أدخل 4 صور لتكوين كلمة السر.",
#                                         font=("Arial", 18, "bold"), bg="#79AAEA", fg="#225E9E",
#                                         pady=10, relief=tk.RAISED, bd=5) # <--- تم التعديل هنا: relief و bd
#         # تم تغيير الصف إلى 4 (كان 1) وزيادة pady لتنزل للأسفل، وتم إزالة "ew"
#         self.instruction_label.grid(row=4, column=0, pady=(150,10), sticky="n") # pady(علوي, سفلي)

#         self.all_predictions_label = tk.Label(self.master, text="التوقعات المتتالية: []",
#                                             font=("Arial", 14), bg="#79AAEA",
#                                             relief=tk.SUNKEN, bd=3) # <--- تم التعديل هنا: relief و bd
#         # تم تغيير الصف إلى 5 (كان 3) وزيادة pady لتنزل للأسفل، وتم إزالة "ew"
#         self.all_predictions_label.grid(row=5, column=0, pady=5, sticky="n")

#         self.upload_button = tk.Button(self.master, text="اختر الصورة", command=self.upload_image,
#                                     font=("Arial", 14, "bold"),
#                                     relief=tk.RAISED, bd=4, cursor="hand2") # <--- تم التعديل هنا: relief و bd وإضافة cursor
#         # تم تغيير الصف إلى 6 (كان 4) وزيادة pady لتنزل للأسفل
#         self.upload_button.grid(row=6, column=0, pady=(20, 30), sticky="n")

# # ملاحظة: لكي تظهر هذه التغييرات بشكل جيد،


#     def upload_image(self):
#         file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png *.gif")])
#         if file_path:
#             self.process_image(file_path)

#     def process_image(self, img_path):
#         # تم التعديل هنا لاستخدام النموذجين h1 و d
#         global h1, d  # التأكد من الوصول للنماذج العامة
#         try:
#             # --- تجهيز الصورة ---
#             img = tf.keras.preprocessing.image.load_img(img_path, target_size=(64, 64), color_mode='grayscale')
#             img_array = tf.keras.preprocessing.image.img_to_array(img)
#             img_array = np.expand_dims(img_array, axis=0) / 255.0
            
#             prediction = None

#             # --- التنبؤ بالصورة بناءً على المرحلة الحالية ---
#             if self.stage == "entering_sequence":
#                 # استخدام النموذج h1 للتحقق من كلمة السر
#                 prediction = np.argmax(h1.predict(img_array, verbose=0))
#             elif self.stage == "awaiting_final_image":
#                 # استخدام النموذج d للتحقق من الصورة النهائية
#                 prediction = np.argmax(d.predict(img_array, verbose=0))
            
#             # --- تخزين آخر تنبؤ في متغير ---
#             self.final_prediction_result = prediction

#             # --- منطق الحالة بناءً على التنبؤ ---
#             if self.stage == "entering_sequence":
#                 self.handle_sequence_stage(prediction)
#             elif self.stage == "awaiting_final_image":
#                 self.handle_final_image_stage(prediction)

#         except NameError as ne:
#             # خطأ في حال عدم تعريف أحد النماذج
#             self.instruction_label.config(text="خطأ: النموذج h1 أو d غير معرف.", fg="red")
#             print(f"Error: Model not found - {ne}")
#         except Exception as e:
#             print(f"خطأ في معالجة الصورة: {e}")

#     def handle_sequence_stage(self, prediction):
#         self.predictions.append(prediction)
#         self.all_predictions_label.config(text=f"التوقعات المتتالية: {self.predictions}")
#         self.current_image_index += 1

#         if self.current_image_index >= 4:
#             self.check_sequences()

#     def handle_final_image_stage(self, prediction):
#         # التحقق من الشرط النهائي بعد إدخال كلمة السر الصحيحة
#         if self.validated_password == 1 and prediction == 3:
#             self.instruction_label.config(text="التحقق صحيح! أهلا بك في المستخدم الاول ...", fg="green")
#             self.master.after(2000, self.launch_app)
#         elif self.validated_password == 2 and prediction == 4:
#             self.instruction_label.config(text="التحقق صحيح! أهلا بك في المستخدم الثاني ...", fg="green")
#             self.master.after(2000, self.launch_app)
#         else:
#             self.instruction_label.config(text=f"فشل التحقق (التوقع كان {prediction}). يرجى المحاولة من جديد.", fg="red")
#             self.master.after(3000, self.reset_state_and_ui)

#     def check_sequences(self):
#         self.upload_button.config(state=tk.DISABLED)
#         if self.predictions == self.password_one_sequence:
#             self.validated_password = 1
#             self.stage = "awaiting_final_image"
#             self.instruction_label.config(text="كلمة السر الأولى صحيحة! الآن أدخل صورة التحقق النهائية.", fg="blue")
#             self.upload_button.config(text="اختر صورة التحقق", state=tk.NORMAL)
#         elif self.predictions == self.password_two_sequence:
#             self.validated_password = 2
#             self.stage = "awaiting_final_image"
#             self.instruction_label.config(text="كلمة السر الثانية صحيحة! الآن أدخل صورة التحقق النهائية.", fg="blue")
#             self.upload_button.config(text="اختر صورة التحقق", state=tk.NORMAL)
#         else:
#             self.instruction_label.config(text="تسلسل خاطئ. سيتم إعادة الضبط.", fg="red")
#             self.master.after(3000, self.reset_state_and_ui)

#     def reset_state_and_ui(self):
#         self.reset_state()
#         self.instruction_label.config(text="أدخل 4 صور لتكوين كلمة السر.", fg="black") # إعادة اللون إلى الأسود الافتراضي
#         self.all_predictions_label.config(text="التوقعات المتتالية: []")
#         self.upload_button.config(text="اختر الصورة", state=tk.NORMAL)

#     def reset_state(self):
#         """إعادة تعيين متغيرات الحالة المنطقية"""
#         self.predictions = []
#         self.current_image_index = 0
#         self.stage = "entering_sequence"
#         self.validated_password = 0  # 0: none, 1: pass1, 2: pass2
#         self.final_prediction_result = None # المتغير لتخزين نتيجة التنبؤ النهائية

#     def launch_app(self):
#         self.master.destroy()
#         root = tk.Tk()
#         if self.validated_password == 1:
#             app = self.main_app_class(root)
#         elif self.validated_password == 2:
#             app = self.third_app_class(root)
        
#         # التأكد من وجود كائن 'app' قبل استدعاء البروتوكول
#         if 'app' in locals() and hasattr(app, 'on_closing'):
#             root.protocol("WM_DELETE_WINDOW", app.on_closing)
#         root.mainloop()

#     def on_closing(self):
#         self.master.destroy()
        
        
# # if __name__ == "__main__":
# #     root_initial = tk.Tk()
# #     # تمرير كل من MainApp و ThirdApp إلى الواجهة الأولية
# #     initial_app = InitialApp(root_initial, MainApp, MainApp2)
# #     root_initial.mainloop()


In [5]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
import tensorflow as tf

# ملاحظة: يجب أن تكون النماذج محملة عالمياً أو تمريرها للكلاس
# h1 = tf.keras.models.load_model('path_to_model_1')
# d = tf.keras.models.load_model('path_to_model_2')

class InitialApp:
    def __init__(self, master, main_app_class, third_app_class):
        self.master = master
        self.main_app_class = main_app_class
        self.third_app_class = third_app_class 

        # --- تعريف كلمات السر ---
        self.password_one_sequence = [2, 4, 3, 2]
        self.password_two_sequence = [1, 5, 0, 1]

        self.master.title("الواجهة الأولية - التحقق من الدخول")
        self.master.geometry("800x600")
        
        # إدارة الحالة
        self.reset_state()

        # إعداد الشبكة (Grid Configuration) لضمان التوسط
        self.master.grid_columnconfigure(0, weight=1)
        # تخصيص أوزان الصفوف لترك مساحة للخلفية وتوسط العناصر
        for i in range(7):
            self.master.grid_rowconfigure(i, weight=1)

        # --- إعداد الواجهة ---
        self._setup_ui()
        self.master.protocol("WM_DELETE_WINDOW", self.on_closing)

    def _setup_ui(self):
        # 1. إعداد صورة الخلفية باستخدام place (لا تتعارض مع grid للعناصر الأخرى)
        background_image_path = 'C:/Users/User/Downloads/pexels-kindelmedia-8566526.jpg'
        try:
            original_image = Image.open(background_image_path)
            resized_image = original_image.resize((800, 600), Image.LANCZOS)
            self.background_photo = ImageTk.PhotoImage(resized_image)
            
            self.background_label = tk.Label(self.master, image=self.background_photo)
            self.background_label.place(x=0, y=0, relwidth=1, relheight=1)
        except Exception as e:
            print(f"تنبيه: تعذر تحميل الخلفية، سيتم استخدام لون سادة. الخطأ: {e}")
            self.master.configure(bg="#79AAEA")

        # 2. إعداد العناصر باستخدام grid حصراً
        # ملحوظة: أضفنا bg مطابقة أو قريبة لجعل النصوص واضحة فوق الخلفية
        
        self.instruction_label = tk.Label(
            self.master, text="أدخل 4 صور لتكوين كلمة السر.",
            font=("Arial", 18, "bold"), bg="#79AAEA", fg="#225E9E",
            pady=10, relief=tk.RAISED, bd=5
        )
        self.instruction_label.grid(row=1, column=0, pady=(100, 10), sticky="n")

        self.all_predictions_label = tk.Label(
            self.master, text="التوقعات المتتالية: []",
            font=("Arial", 14), bg="#79AAEA",
            relief=tk.SUNKEN, bd=3
        )
        self.all_predictions_label.grid(row=2, column=0, pady=5, sticky="n")

        self.upload_button = tk.Button(
            self.master, text="اختر الصورة", command=self.upload_image,
            font=("Arial", 14, "bold"), relief=tk.RAISED, bd=4, cursor="hand2"
        )
        self.upload_button.grid(row=3, column=0, pady=(20, 30), sticky="n")

        # ليبل لعرض معاينة الصورة المرفوعة حالياً
        self.image_display_label = tk.Label(self.master, bg="#79AAEA")
        self.image_display_label.grid(row=4, column=0, pady=10)

    def upload_image(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png *.gif")])
        if file_path:
            # عرض معاينة للصورة
            img_preview = Image.open(file_path).resize((100, 100), Image.LANCZOS)
            img_tk = ImageTk.PhotoImage(img_preview)
            self.image_display_label.config(image=img_tk)
            self.image_display_label.image = img_tk
            
            self.process_image(file_path)

    def process_image(self, img_path):
        global h1, d
        try:
            # التحقق من وجود النماذج قبل البدء
            if 'h1' not in globals() or 'd' not in globals():
                self.instruction_label.config(text="خطأ: النماذج الذكية غير محملة في الذاكرة", fg="red")
                return

            img = tf.keras.preprocessing.image.load_img(img_path, target_size=(64, 64), color_mode='grayscale')
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0
            
            if self.stage == "entering_sequence":
                prediction = np.argmax(h1.predict(img_array, verbose=0))
                self.handle_sequence_stage(prediction)
            elif self.stage == "awaiting_final_image":
                prediction = np.argmax(d.predict(img_array, verbose=0))
                self.handle_final_image_stage(prediction)

        except Exception as e:
            self.instruction_label.config(text=f"خطأ في المعالجة: {str(e)[:30]}...", fg="red")

    def handle_sequence_stage(self, prediction):
        self.predictions.append(prediction)
        self.all_predictions_label.config(text=f"التوقعات المتتالية: {self.predictions}")
        self.current_image_index += 1

        if self.current_image_index >= 4:
            self.check_sequences()

    def handle_final_image_stage(self, prediction):
        if (self.validated_password == 1 and prediction == 3) or \
           (self.validated_password == 2 and prediction == 4):
            self.instruction_label.config(text="تم التحقق بنجاح! جاري الدخول...", fg="green")
            self.master.after(2000, self.launch_app)
        else:
            self.instruction_label.config(text=f"صورة التحقق خاطئة ({prediction}). أعد المحاولة.", fg="red")
            self.master.after(3000, self.reset_state_and_ui)

    def check_sequences(self):
        self.upload_button.config(state=tk.DISABLED)
        if self.predictions == self.password_one_sequence:
            self.validated_password = 1
            self.stage = "awaiting_final_image"
            self.instruction_label.config(text="كلمة السر 1 صحيحة! أدخل صورة التحقق.", fg="blue")
            self.upload_button.config(text="اختر صورة التحقق النهائية", state=tk.NORMAL)
        elif self.predictions == self.password_two_sequence:
            self.validated_password = 2
            self.stage = "awaiting_final_image"
            self.instruction_label.config(text="كلمة السر 2 صحيحة! أدخل صورة التحقق.", fg="blue")
            self.upload_button.config(text="اختر صورة التحقق النهائية", state=tk.NORMAL)
        else:
            self.instruction_label.config(text="تسلسل الصور خاطئ. إعادة ضبط...", fg="red")
            self.master.after(3000, self.reset_state_and_ui)

    def reset_state_and_ui(self):
        self.reset_state()
        self.instruction_label.config(text="أدخل 4 صور لتكوين كلمة السر.", fg="#225E9E")
        self.all_predictions_label.config(text="التوقعات المتتالية: []")
        self.upload_button.config(text="اختر الصورة", state=tk.NORMAL)
        self.image_display_label.config(image='')

    def reset_state(self):
        self.predictions = []
        self.current_image_index = 0
        self.stage = "entering_sequence"
        self.validated_password = 0 
        self.final_prediction_result = None

    def launch_app(self):
        # تدمير النافذة الحالية وفتح التطبيق المطلوب
        self.master.destroy()
        new_root = tk.Tk()
        if self.validated_password == 1:
            self.main_app_class(new_root)
        else:
            self.third_app_class(new_root)
        new_root.mainloop()

    def on_closing(self):
        self.master.destroy()

In [6]:
if __name__ == "__main__":
    root_initial = tk.Tk()
    # تمرير كل من MainApp و ThirdApp إلى الواجهة الأولية
    initial_app = InitialApp(root_initial, MainApp, MainApp2)
    root_initial.mainloop()